# Method 2 completion 1: validation
Attach the strict completion bundle. See docs/method2_completion.md for the required previous outputs. GPU T4, fresh session. Test results must not select hyperparameters.


In [1]:
from pathlib import Path
import os, shutil, subprocess, sys, json
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
bundles = list(Path("/kaggle/input").rglob("completion_manifest.json"))
if len(bundles) != 1:
    raise RuntimeError(f"Attach exactly one completion bundle; found {len(bundles)}")
bundle = bundles[0].parent
work = Path("/kaggle/working")
for name in ["src", "scripts", "configs", "data", "docs", "notebooks"]:
    shutil.copytree(bundle / name, work / name, dirs_exist_ok=True)
shutil.copy2(bundle / "same_domain_feasibility.json", work / "same_domain_feasibility.json")
shutil.copy2(bundle / "completion_manifest.json", work / "completion_manifest.json")
os.chdir(work)
caches = list(Path("/kaggle/input").rglob("models--BAAI--bge-m3"))
if caches:
    cache_hub = caches[0].parent
    os.environ["HF_HUB_CACHE"] = str(cache_hub)
    os.environ["HF_HOME"] = str(cache_hub.parent)
pins = json.loads(Path("configs/method2/pinned_versions.json").read_text())["pinned"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{k}=={v}" for k,v in pins.items()], "jsonschema", "pyyaml", "matplotlib", "rank_bm25", "datasets", "accelerate"], check=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.5 MB/s eta 0:00:00


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers==5.15.1', 'sentence-transformers==6.0.0', 'peft==0.20.0', 'jsonschema', 'pyyaml', 'matplotlib', 'rank_bm25', 'datasets', 'accelerate'], returncode=0)

In [2]:
prior_stage = None
if prior_stage:
    markers = list(Path("/kaggle/input").rglob(f"{prior_stage}_completed.json"))
    if len(markers) != 1:
        raise RuntimeError(f"Attach exactly one strict {prior_stage} output, including results/; found {len(markers)}")
    previous = markers[0].parents[3]
    for name in ["artifacts/method2", "results/method2/completion", "data/method2/index"]:
        if (previous / name).exists():
            shutil.copytree(previous / name, work / name, dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint-*"))
if False:
    validation_markers = list(Path("/kaggle/input").rglob("validation_completed.json"))
    if len(validation_markers) != 1:
        raise RuntimeError("Attach exactly one validation output for the selected CE checkpoint")
    validation_root = validation_markers[0].parents[3]
    shutil.copytree(validation_root / "results/method2/completion/validation", work / "results/method2/completion/validation", dirs_exist_ok=True)
    shutil.copy2(validation_markers[0], work / "results/method2/completion/validation_completed.json")
    shutil.copytree(validation_root / "artifacts/method2/crossencoder", work / "artifacts/method2/crossencoder", dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint-*"))
ce = Path("artifacts/method2/crossencoder/run01/final")
if True and not (ce / "crossencoder_heads.pt").exists():
    heads = [p for p in Path("/kaggle/input").rglob("crossencoder_heads.pt") if p.parent.name == "final"]
    if len(heads) != 1:
        raise RuntimeError(f"Attach one CE final checkpoint (model + heads + tokenizer), found {len(heads)}")
    shutil.copytree(heads[0].parent, ce, dirs_exist_ok=True)


In [3]:
subprocess.run([sys.executable, "scripts/method2/run_completion.py", "validation"], check=True)
print(Path("results/method2/completion/validation_completed.json").read_text(encoding="utf-8")[:2000])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2422.15it/s]


{
  "passed": false,
  "failures": [
    {
      "metric": "enum_accuracy",
      "value": 0.7571,
      "required": 0.9
    },
    {
      "metric": "argument_em",
      "value": 0.45652173913043476,
      "required": 0.7
    }
  ],
  "missing_metrics": [],
  "complete": true,
  "measured_on": "custom_val_seen+custom_val_unseen",
  "inference_diagnostics": {
    "argument_em": 0.45652173913043476,
    "gold_call_count": 460,
    "correct_call_count": 210,
    "values_on_gold": {
      "string": {
        "correct": 598,
        "support": 680,
        "accuracy": 0.8794117647058823
      },
      "required": {
        "correct": 764,
        "support": 947,
        "accuracy": 0.8067581837381204
      },
      "integer": {
        "correct": 185,
        "support": 283,
        "accuracy": 0.6537102473498233
      },
      "boolean": {
        "correct": 90,
        "support": 91,
        "accuracy": 0.989010989010989
      },
      "number": {
        "correct": 47,
        "support"

In [4]:
archive = work / "method2_completion_validation_reports.tar.gz"
items = [name for name in ["results/method2/completion", "data/method2/biencoder/train_mined.jsonl"] if Path(name).exists()]
subprocess.run(["tar", "czf", str(archive), *items], check=True)
print(archive, archive.stat().st_size)
print("Save the whole notebook output for the next stage; this small archive does not contain model weights or the index.")


/kaggle/working/method2_completion_validation_reports.tar.gz 453050
Save the whole notebook output for the next stage; this small archive does not contain model weights or the index.
